# Week 4 · Deutsch-Jozsa and phase kickback

**MIT xPRO QCFx1 · Week 4 · Video Q1M4V3**

This notebook walks through the Deutsch-Jozsa argument and gives you a safe
scaffold for the lab. It **does not** contain the four N=1 oracles. You write
those by hand and commit them only after the lab is submitted.

Time reference: **Paramaribo (UTC−3)**. Hard deadline: **Mon 2026-11-02 20:30**.

## 1. The problem and the promise

An oracle computes a Boolean function

$$
f:\{0,1\}^N \to \{0,1\}.
$$

We are promised that $f$ is either **constant** (same output for every input) or
**balanced** (0 on exactly half the inputs and 1 on the other half). The task is
to decide which.

Classically, a deterministic algorithm may need

$$
\frac{2^N}{2} + 1 = 2^{N-1} + 1
$$

queries in the worst case: it can see $2^{N-1}$ equal outputs and still not know
whether the next input would differ. Deutsch-Jozsa answers with **one** oracle
query.

The N=1 case is the smallest instance. There are only four possible functions,
written as $f(0)f(1)$:

| Function | $f(0)$ | $f(1)$ | Promise |
|---|---:|---:|---|
| `00` | 0 | 0 | constant |
| `11` | 1 | 1 | constant |
| `01` | 0 | 1 | balanced |
| `10` | 1 | 0 | balanced |

The watch guide asks you to build each $U_f$ yourself using only `X` and `CNOT`.
That work belongs in the lab, not in this public notebook.

In [ ]:
from pathlib import Path
import sys

def find_week4_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "week-4", Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "week4_helpers.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repo root, week-4/, or week-4/notebooks/."
    )

WEEK4_DIR = find_week4_dir()
if str(WEEK4_DIR) not in sys.path:
    sys.path.insert(0, str(WEEK4_DIR))

print(f"Week 4 folder: {WEEK4_DIR}")

import math
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit
import qiskit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram
import week4_helpers as w4

print(f"Qiskit version: {qiskit.__version__}")

In [ ]:
print("N  worst-case classical queries  quantum queries")
print("-- ----------------------------  ---------------")
for n in range(1, 11):
    print(f"{n:2d}  {w4.classical_dj_queries(n):28d}  {1:15d}")

## 2. The N=1 walkthrough

Start with the data qubit in $|0\rangle$ and the ancilla in $|1\rangle$:

$$
|0\rangle_{\text{data}}|1\rangle_{\text{ancilla}}.
$$

Apply $H$ to both:

$$
|+\rangle|-\rangle
= \frac{1}{\sqrt{2}}\bigl(|0\rangle+|1\rangle\bigr)
  \frac{1}{\sqrt{2}}\bigl(|0\rangle-|1\rangle\bigr).
$$

The oracle acts as

$$
U_f|x,y\rangle = |x, y \oplus f(x)\rangle.
$$

Because the ancilla is in $|-\rangle$,

$$
U_f|x,-\rangle = (-1)^{f(x)}|x,-\rangle.
$$

The Boolean value has become a **relative phase** on the data qubit. This is
**phase kickback**: the phase appears on the control (data) qubit even though the
oracle wrote into the target (ancilla).

A final Hadamard on the data qubit converts that phase into a computational-basis
outcome. Measuring the data qubit tells constant from balanced; the ancilla ends
in $|1\rangle$.

## 3. Phase kickback without an oracle

To see the mechanism without constructing any Deutsch-Jozsa oracle, use a
controlled phase gate with a non-Boolean angle $\theta$:

$$
|+\rangle|-\rangle
\longrightarrow
\frac{1}{\sqrt{2}}\left(|0\rangle + e^{i\theta}|1\rangle\right)|-\rangle.
$$

A controlled-phase gate of angle $2\theta$ produces a kickback phase
$e^{i\theta}$ on the data branch; the factor of two comes from the ancilla being
in $|-\rangle$ rather than $|0\rangle$ or $|1\rangle$.

For a Boolean oracle, $\theta = \pi f(x)$, and
$e^{i\pi f(x)} = (-1)^{f(x)}$. This cell is a mechanism demo, **not** one of the
four lab oracles.

The printed bitstrings are in Qiskit order: the **leftmost** character is
`q[1]` (the ancilla) and the **rightmost** is `q[0]` (the data). So `01` means
ancilla 0, data 1.

In [ ]:
kickback_phase = math.pi / 3
circuit = QuantumCircuit(2)
circuit.x(1)        # ancilla starts at |1>
circuit.h(1)        # ancilla -> |-
circuit.h(0)        # data -> |+
# A gate angle of 2*kickback_phase gives a kickback phase of kickback_phase.
circuit.cp(2 * kickback_phase, 0, 1)

print(f"Requested kickback phase theta = pi/3 = {kickback_phase:.6f}")
print(f"Controlled-phase gate angle = 2*theta = {2 * kickback_phase:.6f}")
w4.print_statevector(circuit)
print("\nThe two data branches now differ by a phase; the ancilla stays in |->.")

## 4. The Deutsch-Jozsa wrapper

The helper `deutsch_jozsa_circuit(oracle, n)` expects an oracle on $n+1$ qubits:
data qubits `0 .. n-1` and the ancilla on qubit `n`. It builds

1. $X$ on the ancilla,
2. $H$ on every qubit,
3. your oracle,
4. $H$ on the data qubits,
5. measurement of the data qubits.

It never creates the oracle for you.

The next cell is intentionally **lab-gated**. Leave `oracle_n1 = None` until you
have written and submitted your own oracle. Then replace `None` with a
`QuantumCircuit(2)` that implements one of the four functions.

In [ ]:
# LAB-GATED CELL
# After submitting the lab, replace None with your own 2-qubit oracle circuit.
# Example shape (do not fill this in before the lab):
#   oracle_n1 = QuantumCircuit(2)
#   oracle_n1.cx(...)   # your X/CNOT construction
oracle_n1 = None

if oracle_n1 is None:
    print("No oracle supplied yet. This is intentional: write your own after the lab.")
    print("The wrapper is ready: w4.deutsch_jozsa_circuit(oracle_n1, 1).")
else:
    dj = w4.deutsch_jozsa_circuit(oracle_n1, 1)
    display(dj.draw(output="text"))
    counts = w4.run_counts(dj, shots=1024, seed=42)
    display(plot_histogram(counts, title="Deutsch-Jozsa N=1"))

## 5. Checking an oracle you wrote

Two helpers are available after the lab:

- `w4.oracle_truth_table(oracle, n)` probes every basis input and returns
  $(x, f(x))$ pairs. It raises an error if the oracle is not deterministic.
- `w4.is_constant_or_balanced(table)` classifies the table.

They are checkers, not generators. The intended workflow is:

```python
oracle_n1 = QuantumCircuit(2)
# ... your own X and CNOT gates ...
table = w4.oracle_truth_table(oracle_n1, 1)
print(table)
print(w4.is_constant_or_balanced(table))
```

Then run `w4.deutsch_jozsa_circuit(oracle_n1, 1)` and confirm that the measured
data bit agrees with the classification.

In [ ]:
help(w4.oracle_truth_table)
print("\n" + "=" * 70 + "\n")
help(w4.is_constant_or_balanced)

## 6. What the final interference does

After phase kickback, the data register is in

$$
\frac{1}{\sqrt{2^N}}\sum_x (-1)^{f(x)}|x\rangle.
$$

Applying $H^{\otimes N}$ gives an amplitude on $|0\rangle^{\otimes N}$ equal to

$$
\frac{1}{2^N}\sum_x (-1)^{f(x)}.
$$

- If $f$ is constant, the sum is $\pm 2^N$, so the all-zero outcome has
  probability 1.
- If $f$ is balanced, the sum is 0, so the all-zero outcome has probability 0.

For N=1, "not zero" means the data qubit is measured as `1`. The video's
"exponential" wording is about the classical worst-case query count; the
advantage only becomes meaningful as $N$ grows.

## 7. Predict before you run

1. For each of the four N=1 functions, what should the data qubit read?
2. What does the ancilla read at the end, before it is discarded?
3. Why does the oracle not need to be queried more than once?
4. If the promise is broken (neither constant nor balanced), what does the
   interference argument tell you? (The algorithm's guarantee no longer holds.)

Do the four N=1 constructions on paper or in a private file first. Commit them
only after the lab is submitted.